# LogiCrisis — GRPO Training Notebook

**Meta PyTorch OpenEnv Hackathon**

This notebook trains a Llama-3-8B-Instruct LoRA adapter on the LogiCrisis multi-agent logistics crisis environment using **GRPO (Group Relative Policy Optimization)**.

### Setup
1. Runtime → Change runtime type → **T4 GPU** (free tier) or A100 (Colab Pro+)
2. Add your HuggingFace token in the `HF_TOKEN` cell below
3. Run all cells

### Links
- [HF Space (live environment)](https://huggingface.co/spaces/Sana06112003/logicriasis-training)
- [Trained adapter](https://huggingface.co/Sana06112003/logicriasis-adapter)
- [GitHub repo](https://github.com/SANGRAMLEMBE/logicriasis)

In [ ]:
# Check GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], 
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'No GPU detected')

In [ ]:
# Install dependencies
import sys
!{sys.executable} -m pip install -q \
    'torch>=2.5.0' torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu124

!{sys.executable} -m pip install -q \
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' \
    'trl>=0.9.0' \
    datasets \
    accelerate \
    peft \
    bitsandbytes \
    matplotlib \
    requests \
    fastapi \
    pydantic

print('Dependencies installed.')

In [ ]:
# Clone the LogiCrisis repository
!git clone https://github.com/SANGRAMLEMBE/logicriasis /content/logicriasis
%cd /content/logicriasis/logicriasis
import sys
sys.path.insert(0, '/content/logicriasis/logicriasis')
print('Repository cloned.')

In [ ]:
import os

# ── Set your HuggingFace token here ──────────────────────────────────────────
HF_TOKEN = ""   # <-- paste your HF write token here (hf_...)
MODEL_REPO = "Sana06112003/logicriasis-adapter"   # where to push the trained adapter
# ─────────────────────────────────────────────────────────────────────────────

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["MODEL_REPO"] = MODEL_REPO

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print(f'Logged in. Adapter will be pushed to: {MODEL_REPO}')
else:
    print('No HF_TOKEN set — adapter will be saved locally only.')

In [ ]:
import torch

# Shim for older torch builds
if not hasattr(torch.utils._pytree, 'register_constant'):
    torch.utils._pytree.register_constant = lambda cls: cls

if not torch.cuda.is_available():
    print('WARNING: No GPU — training will be very slow on CPU.')
    GPU_TIER = 't4'
else:
    gpu_name = torch.cuda.get_device_name(0).lower()
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram_gb:.1f} GB')
    if 'a100' in gpu_name or 'l40' in gpu_name or 'h100' in gpu_name or vram_gb >= 40:
        GPU_TIER = 'a100'
    elif 'a10' in gpu_name or vram_gb >= 20:
        GPU_TIER = 'a10g'
    else:
        GPU_TIER = 't4'

print(f'GPU Tier: {GPU_TIER.upper()}')

# Hyperparameters per GPU tier
if GPU_TIER == 'a100':
    BASE_MODEL = 'unsloth/llama-3-8b-instruct-bnb-4bit'
    MAX_SEQ_LEN = 8192; LORA_R = 64; LORA_ALPHA = 64
    LOAD_IN_4BIT = True; USE_BF16 = True
    BATCH_SIZE = 4; GRAD_ACC = 2; NUM_GENERATIONS = 16
    MAX_COMPLETION = 512; NUM_EPOCHS = 5; LR = 3e-5
elif GPU_TIER == 'a10g':
    BASE_MODEL = 'unsloth/llama-3-8b-instruct-bnb-4bit'
    MAX_SEQ_LEN = 4096; LORA_R = 32; LORA_ALPHA = 32
    LOAD_IN_4BIT = True; USE_BF16 = True
    BATCH_SIZE = 2; GRAD_ACC = 4; NUM_GENERATIONS = 16
    MAX_COMPLETION = 384; NUM_EPOCHS = 4; LR = 4e-5
else:  # T4
    BASE_MODEL = 'unsloth/llama-3-8b-instruct-bnb-4bit'
    MAX_SEQ_LEN = 2048; LORA_R = 16; LORA_ALPHA = 16
    LOAD_IN_4BIT = True; USE_BF16 = False
    BATCH_SIZE = 1; GRAD_ACC = 8; NUM_GENERATIONS = 8
    MAX_COMPLETION = 256; NUM_EPOCHS = 3; LR = 5e-5

OUTPUT_DIR = '/content/logicriasis_outputs'
ADAPTER_DIR = f'{OUTPUT_DIR}/final'

print(f'Config: max_seq={MAX_SEQ_LEN} | lora_r={LORA_R} | batch={BATCH_SIZE} | '
      f'grad_acc={GRAD_ACC} | epochs={NUM_EPOCHS} | bf16={USE_BF16}')

In [ ]:
import time
from unsloth import FastLanguageModel

print(f'Loading {BASE_MODEL}...')
t0 = time.time()
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=torch.bfloat16 if USE_BF16 else None,
    load_in_4bit=LOAD_IN_4BIT,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
print(f'Model loaded in {time.time()-t0:.1f}s')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
from training.train import build_curriculum_dataset
from collections import Counter

warmup_samples = 1024 if GPU_TIER in ('a100', 'a10g') else 512

print(f'Building 6-role curriculum dataset (warmup={warmup_samples})...')
dataset = build_curriculum_dataset(warmup_samples=warmup_samples)
role_counts = Counter(dataset['role'])
print(f'Total prompts: {len(dataset)}')
for role, cnt in sorted(role_counts.items()):
    print(f'  {role:<25} {cnt} prompts')

In [ ]:
from trl import GRPOConfig, GRPOTrainer
from training.train import grpo_reward_fn

print(f'Starting GRPO training ({GPU_TIER.upper()} config)...')

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACC,
    learning_rate=LR,
    logging_steps=5,
    save_steps=50,
    report_to='none',
    max_completion_length=MAX_COMPLETION,
    num_generations=NUM_GENERATIONS,
    temperature=0.8,
    seed=42,
    bf16=USE_BF16,
    fp16=(not USE_BF16),
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[grpo_reward_fn],
    args=grpo_config,
    train_dataset=dataset,
)

t_train = time.time()
trainer.train()
elapsed = time.time() - t_train
print(f'Training complete in {elapsed/60:.1f} min')

In [ ]:
import os
from training.train import save_training_curves

print(f'Saving LoRA adapter to {ADAPTER_DIR}')
os.makedirs(ADAPTER_DIR, exist_ok=True)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

os.makedirs('assets', exist_ok=True)
save_training_curves(trainer.state.log_history, output_dir='assets')
print('Training curves saved to assets/')

# Display training curves
import matplotlib.pyplot as plt
from IPython.display import display, Image
if os.path.exists('assets/training_curves.png'):
    display(Image('assets/training_curves.png'))

In [ ]:
if HF_TOKEN:
    print(f'Pushing adapter to hub: {MODEL_REPO}')
    try:
        from huggingface_hub import HfApi
        api = HfApi()
        api.create_repo(repo_id=MODEL_REPO, repo_type='model', exist_ok=True)
        api.upload_folder(
            folder_path=ADAPTER_DIR,
            repo_id=MODEL_REPO,
            repo_type='model',
            commit_message=f'GRPO-trained LogiCrisis ({GPU_TIER.upper()}, r={LORA_R}, {NUM_EPOCHS}ep)',
        )
        print(f'Adapter live at: https://huggingface.co/{MODEL_REPO}')
    except Exception as e:
        print(f'Push failed: {e} — adapter saved locally at {ADAPTER_DIR}')
else:
    print('No HF_TOKEN — adapter saved locally. Set HF_TOKEN to push to hub.')

In [ ]:
# Quick post-training benchmark
print('Running post-training heuristic benchmark...')
FastLanguageModel.for_inference(model)

from inference import run_episode
from environment.tasks import ALL_TASK_IDS

print(f'{"Task":<35} {"Score":>8} {"Status"}')
print('-' * 55)
passed = 0
for task_id in ALL_TASK_IDS:
    result = run_episode(task_id, use_llm=False)
    status = 'PASS' if result['passed'] else 'FAIL'
    if result['passed']:
        passed += 1
    print(f'  {task_id:<33} {result["score"]:>8.4f}  {status}')

print('-' * 55)
print(f'  Pass rate: {passed}/{len(ALL_TASK_IDS)}')

## Training Complete

The trained LoRA adapter is now:
- Saved locally at `/content/logicriasis_outputs/final/`
- Pushed to HuggingFace Hub at [Sana06112003/logicriasis-adapter](https://huggingface.co/Sana06112003/logicriasis-adapter) (if HF_TOKEN was set)

### Using the adapter for inference

```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Sana06112003/logicriasis-adapter",
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)
```

### Evaluating on the live environment

```python
import requests
BASE = "https://Sana06112003-logicriasis-training.hf.space"
obs = requests.post(f"{BASE}/reset", json={"task_id": "cascade_failure_recovery"}).json()
grade = requests.post(f"{BASE}/grade").json()
print(f"Score: {grade['score']}")
```